# Tornado Retrofit Optimizer

Tornado retrofit optimizer interactive tool. This code runs the Alternative for Resilient Communities (ARC) tornado model.

To run interactive dashboard, you can use voila! to hide the underlying code.

User defines tornado event cases and budgets, and the outputs can be displayed through maps, text outputs, graphs, or raw data

The code calls AMPL using Python API to run optimization

The data are contained in a separate file. New tornado events can be generated there

For questions, contact william.hughes@nist.gov or kenneth.harrison@nist.gov

# Set up
Note you may need to edit the ampl path to link to your license

In [1]:
print('Loading packages...')

import subprocess
import sys
import importlib

# List of required packages
required_packages = ["numpy", "pandas", "matplotlib",
                    "amplpy","copy","contextily","geopandas",
                    "ipynb","ipywidgets","networkx",
                    "scipy","openpyxl","glob","shutil"]  # Add your packages here

version = ['2.2.4','2.2.3','3.10.1','0.14.0',None,'1.6.0','1.0.0',None,'8.1.3','3.2.1','1.15.2','3.1.4',None,None]


def install_and_import(package, version=None):
    pkg_name = f"{package}=={version}" if version else package
    try:
        importlib.import_module(package)
        print(f"{package} is already installed.")
    except ImportError:
        print(f"Installing {pkg_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])
    finally:
        globals()[package] = importlib.import_module(package)
        
# Install and import all required packages
i = 0
for package in required_packages:
    install_and_import(package,version[i])
    i+=1
    
# Now you can use the packages directly
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import amplpy
from amplpy import AMPL, ampl_notebook, Environment
import copy
import contextily as cx
import geopandas as gp
import ipynb
from ipywidgets import interact, fixed, widgets, interact_manual
import ipywidgets
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import pickle
import warnings
from amplpy import AMPL, add_to_path

# Create an AMPL instance (update license here)
ampl = AMPL()
# Example of setting up AMPL path
#add_to_path("C:\\your\\path\\to\\AMPL")
#add_to_path("C:\\Users\\username\\AMPL")

# Suppress the specific warning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore")

directory = rf"backend/Data/"

Loading packages...
numpy is already installed.
pandas is already installed.
matplotlib is already installed.
amplpy is already installed.
copy is already installed.
contextily is already installed.
geopandas is already installed.
ipynb is already installed.
ipywidgets is already installed.
networkx is already installed.
scipy is already installed.
openpyxl is already installed.
glob is already installed.
shutil is already installed.


In [2]:
# Call functions
%run backend/run_opt.ipynb 
%run backend/Output_generator.ipynb
%run backend/optimization_pipeline.ipynb
%run backend/post_process_results.ipynb
%run backend/set_up_optimization.ipynb

# Update parameters

In the data folder, parameters can be updated by updating the Community_data file in the Data Folder

For full details, please see the readme file or the Data Description document in the Data folder
To run the analysis with default conditions for Joplin, MO, these steps can be skipped.

To run the analysis for a new community or tornado path beside the default of Joplin, the following steps are needed:

1. The building footprint factors can be updated for your specific region. The parameters for different return periods can be extracted from the [ASCE 7 Hazard Tool]( [https://ascehazardtool.org/) for your region of interest. Update the return_period excel file in the Data Folder accordingly. However, as the footprint factors are based primarily n the buidling size, they can be assumed independent of hazard exposure frequency and kept as default.

2. Update tornado frequencies for your specific region. Specifically, due to limited tornado data on a small community level, a larger scaele, e.g., county, should be used. The historic data is downloaded from SVIRS.

3. Update GIS data for your specific region. Specifically, you will need a new neighborhood shapefile, census data, and tornado path uploaded to the proper folders.

4. Run data_generator script to obtain updated parameters relating to damage state probabilities, displacement probabilities, and casualty rates. After updating the data, these simulations can be re-run below. Otherwise, if using default location of Joplin, MO without adjustments, or if the updates have already been run, proceed to optimization.

In [3]:
# load data

neighborhoods = gp.read_file(rf"{directory}Neighborhoods/neighborhoods.shp")
prj = neighborhoods.crs
data_file = rf"{directory}community_data.xlsx"

In [4]:
# ============== Widget for budget

w1 = widgets.BoundedFloatText(
    value=100,  # Default value
    min=0.0,    # Minimum value (greater than 0)
    max=10000,
    description='Budget ($Million):',
    step=1,   # Step size
    disabled=False,
    style = {'description_width': 'initial'}
)

In [5]:
# ================= Widget for tornado event

w2 = widgets.Dropdown(
    options=['EF0','EF1','EF2','EF3','EF4','EF5','Joplin','Weighted'],
    value='EF3',
    disabled=False,
    description='Event:',
    style = {'description_width': 'initial'}
)

In [6]:
# Widget for whether to fix building stock

w3 = widgets.Checkbox(value=False, description="Fix building stock from last analysis",style={'description_width': '0px'})


# Run optimization

In [7]:
widgets.interact_manual.opts['manual_name'] = 'Run optimization'
interact_manual(optimization_pipeline,event=w2,budget = w1,x_b_fix=w3,
         directory=fixed(directory),manual_name = 'Run optimization');

interactive(children=(Dropdown(description='Event:', index=3, options=('EF0', 'EF1', 'EF2', 'EF3', 'EF4', 'EF5…

# Display results

In [31]:
# Rename widget options
global options
options = ['Text Output',
           'Structural retrofits map',
           'Shelter map',
           'Retrofits and shelter map',
           'All building investments map',
           'Benefit-cost ratio map',
           'Ethnicity map',
           'Renter status map',
           'Population map',
           'Loss breakdown',
           'Investment breakdown',
           'Download data (.shp)']
w4 = widgets.Dropdown(
    options= options,
    description='Output type:',
    disabled=False,
    layout=widgets.Layout(width='280px')
)

In [32]:
my_interact_manual = interact_manual.options(manual_name="Plot result")
my_interact_manual(plot_figures,outputs=w4,directory=fixed(directory))

interactive(children=(Dropdown(description='Output type:', layout=Layout(width='280px'), options=('Text Output…

<function __main__.plot_figures(outputs)>